### Objective:

            To demonstrate why NumPy is faster than standard Python lists for numerical operations, specifically visualizing the memory architectures shown in the diagram.


🚀 Prerequisite Setup
Before starting the demonstration, run this block to import necessary libraries and define our visualization tools.

In [1]:
import sys
import timeit
import numpy as np
import ctypes
from decimal import Decimal


In [2]:

# --- VISUALIZATION HELPER FUNCTION ---
# This function is designed to display the memory layout clearly to users.
# You don't need to explain the code inside, just the OUTPUT it creates.

def visualize_memory(structure, title="Memory Layout"):
    """
    Prints a clear text-based schematic of the memory layout
    of a given Python list or NumPy array.
    """
    print("=" * 60)
    print(f"🔬 {title.upper()} 🔬")
    print("=" * 60)

    # 1. Inspecting a Python List (❌ Scattered)
    if isinstance(structure, list):
        print("\n[OBJECT: Python List (Pointer Array)]")
        print("  |   Index  |  Pointer/Address Living at Index  |")
        print("  |----------|-----------------------------------|")
        
        raw_list_address = id(structure)
        for i, item in enumerate(structure):
            item_address = id(item)
            # Find the address of the POINTER itself within the list array
            # Note: This is an estimation, as id() gives the address of the object, 
            # not its position *inside* another object's structure.
            # We visualize the logical organization.
            
            # The values are references
            pointer_representation = f"Ref to {hex(item_address)}"
            
            print(f"  | Index {i:2} | {hex(item_address):>33} |")
            print("  |__________|___________________________________|")
        
        print("\n  >>> Arrows 'jump' across RAM:")
        for i, item in enumerate(structure):
            item_address = id(item)
            item_type = type(item).__name__.capitalize()
            print(f"      [{hex(item_address)}] ───> ┌──────────────────┐")
            print(f"                           │ Type: {item_type:<6} | V: {item:<2} │")
            print(f"                           └──────────────────┘")

    # 2. Inspecting a NumPy Array (✅ Contiguous)
    elif isinstance(structure, np.ndarray):
        print("\n[OBJECT: NumPy Array (Tightly Packed Block)]")
        print(" [  Metadata Header  ]")
        print(" | (Shape, Dtype, etc.) |")
        print(" └───────────────────┘")
        print("          |")
        print("          v (Contiguous Block starts)")
        
        base_address = structure.ctypes.data
        item_size = structure.itemsize
        
        print("\n  Hex Address   | Tightly Packed Data | Value")
        print("  ------------- | ------------------ | -----")
        
        for i in range(structure.size):
            item_address = base_address + (i * item_size)
            # The addresses are perfectly sequential
            
            # Read raw bytes at that address using ctypes for visualization
            raw_data = ctypes.string_at(item_address, item_size)
            raw_data_hex = raw_data.hex(' ') # format as hex bytes
            
            print(f"  {hex(item_address)}   | [{raw_data_hex:<17}] | {structure[i]}")
            print("  ------------- | ------------------ | -----")
            
        print("\n  >>> Sequential access:")
        # Show sequential pointer visualization
        seq_vis = " ".join([f"[{x}]" for x in structure])
        print(f"      Tightly Packed Sequence: {seq_vis}")
        
    print("\n" + "=" * 60)

print("✅ Setup complete. Visualization tools ready.")

✅ Setup complete. Visualization tools ready.


### Part 1: Visualizing the Python List (The "Scattered" Approach)

We will recreate the exact list used in the diagram: [5, 10, 2].

A standard Python list does not store numbers directly. It stores pointers (addresses) to objects scattered randomly throughout RAM. Each object is heavy, containing metadata like Type: Int.

In [4]:
# 1. Create the example data
demo_py_list = [5, 10, 2]
print(f"Created standard Python List: {demo_py_list}")

# 2. RUN THE VISUALIZATION
# Direct users to compare this output with the top part of the diagram.
visualize_memory(demo_py_list, "Python List (Scattered)")

Created standard Python List: [5, 10, 2]
🔬 PYTHON LIST (SCATTERED) 🔬

[OBJECT: Python List (Pointer Array)]
  |   Index  |  Pointer/Address Living at Index  |
  |----------|-----------------------------------|
  | Index  0 |                    0x7ffedb5b6a38 |
  |__________|___________________________________|
  | Index  1 |                    0x7ffedb5b6ad8 |
  |__________|___________________________________|
  | Index  2 |                    0x7ffedb5b69d8 |
  |__________|___________________________________|

  >>> Arrows 'jump' across RAM:
      [0x7ffedb5b6a38] ───> ┌──────────────────┐
                           │ Type: Int    | V: 5  │
                           └──────────────────┘
      [0x7ffedb5b6ad8] ───> ┌──────────────────┐
                           │ Type: Int    | V: 10 │
                           └──────────────────┘
      [0x7ffedb5b69d8] ───> ┌──────────────────┐
                           │ Type: Int    | V: 2  │
                           └──────────────────┘



### Part 2: Visualizing the NumPy Array (The "Contiguous" Approach)

Example Data:
We recreate the same list using np.array([5, 10, 2]). We are using 64-bit integers (int64), meaning each number takes exactly 8 bytes of memory

NumPy requires all items to be of the exact same type. This sacrifice allows NumPy to request a single, massive, continuous block of RAM. Numbers are packed raw, side-by-side, without any object overhead.

In [5]:
# 1. Create the example data as a NumPy Array
demo_np_arr = np.array([5, 10, 2], dtype=np.int64)
print(f"Created NumPy Array: {demo_np_arr}")

# 2. RUN THE VISUALIZATION
# Direct users to compare this output with the bottom part of the diagram.
visualize_memory(demo_np_arr, "NumPy Array (Contiguous)")

Created NumPy Array: [ 5 10  2]
🔬 NUMPY ARRAY (CONTIGUOUS) 🔬

[OBJECT: NumPy Array (Tightly Packed Block)]
 [  Metadata Header  ]
 | (Shape, Dtype, etc.) |
 └───────────────────┘
          |
          v (Contiguous Block starts)

  Hex Address   | Tightly Packed Data | Value
  ------------- | ------------------ | -----
  0x2a7040e5f10   | [05 00 00 00 00 00 00 00] | 5
  ------------- | ------------------ | -----
  0x2a7040e5f18   | [0a 00 00 00 00 00 00 00] | 10
  ------------- | ------------------ | -----
  0x2a7040e5f20   | [02 00 00 00 00 00 00 00] | 2
  ------------- | ------------------ | -----

  >>> Sequential access:
      Tightly Packed Sequence: [5] [10] [2]



### Part 3: The Performance Payoff (Vectorization Benchmark)

Memory architecture is abstract, but time is real. Now we prove that the contiguous structure leads to massive speed differences.

We will scale the problem from 3 items to 10 million items. The task is simple: add 10 to every number in the sequence.

In [6]:
# Setup larger datasets (10 million items)
# (Warning: A standard Python list of 10M items uses substantial RAM!)
N = 10000000
test_data = list(range(N)) # Seed data
print(f"Datasets of size N={N:,} items initialized.")

# Create the structures we will benchmark
large_py_list = list(test_data)
large_np_arr = np.array(test_data, dtype=np.int64)

Datasets of size N=10,000,000 items initialized.


### 3.1 Performance Timing Test

We define two functions. The Python List method uses a loop (one item at a time), while the NumPy method uses Vectorization (a 'massive sweep').

In [7]:
# --- Define the operations we are timing ---

# 1. Python List (Iterative Loop method - SLOW)
def python_list_add():
    return [x + 10 for x in large_py_list] # This iterates 10M times

# 2. NumPy Array (Vectorized Operation - FAST)
def numpy_array_add():
    return large_np_arr + 10 # This is the "massive sweep" operation

# --- Run the benchmark ---
print(f"🚀 Running speed benchmark (add 10 to {N:,} elements)...")

# Number of iterations to average the timing
iterations = 3

# Time the Python List operation
py_time = timeit.timeit(python_list_add, number=iterations) / iterations
print(f"❌ Python List average time: {py_time:.4f} seconds.")

# Time the NumPy Array operation
np_time = timeit.timeit(numpy_array_add, number=iterations) / iterations
print(f"✅ NumPy Array average time:  {np_time:.4f} seconds.")

🚀 Running speed benchmark (add 10 to 10,000,000 elements)...
❌ Python List average time: 0.7338 seconds.
✅ NumPy Array average time:  0.0294 seconds.


### Conclusion and Presentation View

In [8]:
print("\n" + "=" * 60)
print("📊 FINAL BENCHMARK SUMMARY 📊")
print("=" * 60)

print(f"Operation:  Add 10 to {N:,} elements")
print(f"Python List Time: {py_time:.4f} seconds (❌ Scattered Memory Loops)")
print(f"NumPy Array Time: {np_time:.4f} seconds (✅ Contiguous Memory Vectorization)")

if np_time > 0:
    speedup = py_time / np_time
    print("-" * 60)
    # Using Decimal for clean formatting
    speedup_fmt = Decimal(speedup).quantize(Decimal('1.1'))
    
    print(f"CONCLUSION:")
    print(f"By utilizing contiguous memory, NumPy is approx {speedup_fmt}x FASTER than standard Python.")
else:
    print("\nSpeedup calculation failed (NumPy time too low to measure accurately).")
print("=" * 60)


📊 FINAL BENCHMARK SUMMARY 📊
Operation:  Add 10 to 10,000,000 elements
Python List Time: 0.7338 seconds (❌ Scattered Memory Loops)
NumPy Array Time: 0.0294 seconds (✅ Contiguous Memory Vectorization)
------------------------------------------------------------
CONCLUSION:
By utilizing contiguous memory, NumPy is approx 25.0x FASTER than standard Python.


The 100x+ speedup isn't magic; it is the direct result of the memory architecture visualized in Parts 1 and 2. Contiguous memory allows the hardware to perform a continuous sweep of data, while scattered memory forces the CPU to waste time navigating a maze of pointers.

In [12]:
zeros = np.zeros((3,3))
zeros

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

In [16]:
ones = np.ones((2,4))
ones

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [18]:
random_matrix = np.random.rand(3,3)
random_matrix

array([[0.25217633, 0.59125579, 0.15091014],
       [0.17206205, 0.10692187, 0.30858593],
       [0.11763985, 0.86777825, 0.13399799]])

In [22]:
linear_data = np.arange(1,13)
linear_data 

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [27]:
matrix_3x4 = linear_data.reshape(6,2)
matrix_3x4

array([[ 1,  2],
       [ 3,  4],
       [ 5,  6],
       [ 7,  8],
       [ 9, 10],
       [11, 12]])

In [28]:
tensor_3d = linear_data.reshape(2,2,3)
tensor_3d

array([[[ 1,  2,  3],
        [ 4,  5,  6]],

       [[ 7,  8,  9],
        [10, 11, 12]]])

In [30]:
data = np.array([[80,85,90],
                 [70,75,80],
                [90,95,100]])

bonus = np.array([2,10,5])

score = data + bonus
score

array([[ 82,  95,  95],
       [ 72,  85,  85],
       [ 92, 105, 105]])

In [1]:
!nvidia-smi

Wed Mar 25 08:35:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 561.03                 Driver Version: 561.03         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 Ti   WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   66C    P0             17W /   50W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----